# Logistic Regression with Reduced Dataset

This notebook goes through the preparation and training for finding the best Logistic Regression model on a reduced dataset with the top 1000 genes determined from our t-test.

In [2]:
import pandas as pd
import numpy as np

from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import FastICA

from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.base import clone

from sklearn.model_selection import cross_val_score, StratifiedKFold, GridSearchCV
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, roc_auc_score, roc_curve

import matplotlib.pyplot as plt
import seaborn as sns


In [ ]:
path = ['./Data/X_train.csv', './Data/y_train.csv']
X_train_temp, y_train_temp = [pd.read_csv(f, index_col=0) for f in path]

top_1000_genes_temp = pd.read_csv('./top_1000_genes.csv', index_col=0)

In [ ]:
top_1000_genes = list(top_1000_genes_temp['0'].copy())

In [ ]:
X_train = X_train_temp[top_1000_genes].copy()
y_train= np.array(y_train_temp['Cluster'].copy())

In [6]:
X_train.shape

(610, 1000)

## Single CV

Model created from single cross-validation run to see how well it performs. 

In [7]:
log_pipe = Pipeline([
    ('scaler', StandardScaler()),
    ('model', LogisticRegression(max_iter=500000))
])

num_splits=5
skfold = StratifiedKFold(n_splits=num_splits, shuffle=True, random_state=123)

acc_score_per_split = np.zeros(num_splits)
rmse_per_split = np.zeros(num_splits)
i=0

for train_index, test_index in skfold.split(X_train, y_train):

    X_tt, X_ho = X_train.iloc[train_index], X_train.iloc[test_index]
    y_tt, y_ho = y_train[train_index], y_train[test_index]

    log_pipe.fit(X_tt, y_tt)
    log_pred = log_pipe.predict(X_ho)
    acc_score_per_split[i] = accuracy_score(y_ho, log_pred)

    i=i+1


print('Logistic Regression Results')
print('Accuracy from split:', acc_score_per_split)
print('Mean: ', np.mean(acc_score_per_split), '. Standard Deviation: ', np.std(acc_score_per_split))


Logistic Regression Results
Accuracy from split: [0.95081967 0.85245902 0.90983607 0.90983607 0.91803279]
Mean:  0.9081967213114754 . Standard Deviation:  0.03170340918985856


## Hyperparameter tuning

Finding the best parameters for our model to achieve the best accuracy. 

Starting off with a single cross-validation run, and then one with Nested CV.

In [8]:
# Define pipeline
pipe = Pipeline([
    ('scale', StandardScaler()),
    ('model', LogisticRegression(max_iter=50000))
])


# Define parameter grid for tuning
param_grid = {
    'model__C': [0.001, 0.01, 0.1, 1, 10, 100],  
    'model__penalty': ['l1', 'l2'],              # can test 'l1' if using solver='liblinear'
    'model__solver': ['liblinear', 'saga'],
    'model__max_iter': [200000]
}

# Cross-validation setup
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# GridSearchCV will find the best combo of params
grid_search = GridSearchCV(pipe, param_grid, cv=cv, scoring='accuracy', n_jobs=-1)
grid_search.fit(X_train, y_train)

print("Best parameters found:")
print(grid_search.best_params_)

print("\nBest cross-validation accuracy:")
print(grid_search.best_score_)

Best parameters found:
{'model__C': 0.01, 'model__max_iter': 200000, 'model__penalty': 'l2', 'model__solver': 'saga'}

Best cross-validation accuracy:
0.9163934426229507


### Nested CV for Hyperparameter tuning

In [9]:
# Define pipeline
pipe = Pipeline([
    ('scale', StandardScaler()),
    ('model', LogisticRegression(max_iter=50000))
])


# Define parameter grid for tuning
param_grid = {
    'model__C': [0.001, 0.01, 0.1, 1, 10, 100],  # inverse regularization strength
    'model__penalty': ['l1', 'l2'],              # can test 'l1' if using solver='liblinear'
    'model__solver': ['liblinear', 'saga'],
    'model__max_iter': [200000]
}

# 2) Nested CV
outer_cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=123)
nested_scores_auc = []
nested_scores_acc = []

inner_cv_scores = []
best_params_list = []


X = X_train.copy()
y = y_train.copy()

i=1
for train_idx, test_idx in outer_cv.split(X, y):
    X_outer_train, X_outer_test = X.iloc[train_idx], X.iloc[test_idx]
    y_outer_train, y_outer_test = y[train_idx], y[test_idx]
    
    # Inner CV for hyperparameter tuning
    inner_cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)
    grid_search = GridSearchCV(pipe, param_grid, cv=inner_cv, scoring='accuracy', n_jobs=-1)
    grid_search.fit(X_train, y_train)

    inner_cv_scores.append(float(grid_search.best_score_))
    best_params_list.append(grid_search.best_params_)

    
    # Evaluate best model on outer test fold
    best_inner_model = clone(grid_search.best_estimator_)
    best_inner_model.fit(X_outer_train, y_outer_train)

    y_outer_proba = best_inner_model.predict_proba(X_outer_test)[:, 1]
    auc = roc_auc_score(y_outer_test, y_outer_proba)
    nested_scores_auc.append(auc)

    acc = accuracy_score(y_outer_test, best_inner_model.predict(X_outer_test))
    nested_scores_acc.append(acc)

    print(f'Fold {i} complete')
    i=i+1

print("\nNested CV (unbiased estimate):")
print("Inner fold accuracies:", inner_cv_scores)
print("Outer fold accuracies:", nested_scores_acc)
print(f"Mean Accuracy: {np.mean(nested_scores_acc)} ± {np.std(nested_scores_acc)}")

print("Outer fold ROC-AUC:", nested_scores_auc)
print(f"Mean ROC-AUC: {np.mean(nested_scores_auc)} ± {np.std(nested_scores_auc)}")

# Summarize best parameters across folds
best_params_df = pd.DataFrame(best_params_list)
print("\nBest parameters per outer fold:")
print(best_params_df)

# Choose the most frequent or best-performing parameter set
final_best_params = best_params_df.mode().iloc[0].to_dict()
print("\nFinal chosen parameters for full training:")
print(final_best_params)



Fold 1 complete
Fold 2 complete
Fold 3 complete
Fold 4 complete
Fold 5 complete

Nested CV (unbiased estimate):
Inner fold accuracies: [0.9197092630155511, 0.9197092630155511, 0.9197092630155511, 0.9197092630155511, 0.9197092630155511]
Outer fold accuracies: [0.9344262295081968, 0.9016393442622951, 0.9098360655737705, 0.9016393442622951, 0.8934426229508197]
Mean Accuracy: 0.9081967213114754 ± 0.014102172568922353
Outer fold ROC-AUC: [0.9822660098522167, 0.9517241379310346, 0.9702842377260982, 0.9677002583979328, 0.9741602067183461]
Mean ROC-AUC: 0.9692269701251256 ± 0.010041425980226832

Best parameters per outer fold:
   model__C  model__max_iter model__penalty model__solver
0       0.1           200000             l1          saga
1       0.1           200000             l1          saga
2       0.1           200000             l1          saga
3       0.1           200000             l1          saga
4       0.1           200000             l1          saga

Final chosen parameters 